In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch

assert torch.cuda.is_available(), ' No GPU found! Switch to A100 runtime.'

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU ready | VRAM: {total_vram:.1f} GB')

NVIDIA A100-SXM4-40GB, 40960 MiB
GPU ready | VRAM: 42.4 GB


In [3]:
import os
import pandas as pd
import time
import json

In [4]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

Mounted at /content/drive
Drive mounted


In [5]:

MODEL_PATH = '/content/drive/MyDrive/models/Qwen3.6-35B-A3B-GGUF/Qwen3.6-35B-A3B-UD-Q5_K_M.gguf'
assert os.path.exists(MODEL_PATH), f'❌ Model not found at: {MODEL_PATH}'
size_gb = os.path.getsize(MODEL_PATH) / 1e9
print(f'✅ Model found: {size_gb:.1f} GB')
print(f'   Path: {MODEL_PATH}')

✅ Model found: 26.5 GB
   Path: /content/drive/MyDrive/models/Qwen3.6-35B-A3B-GGUF/Qwen3.6-35B-A3B-UD-Q5_K_M.gguf


In [6]:
print('🔧 Installing llama-cpp-python with CUDA support...')

!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir -q


🔧 Installing llama-cpp-python with CUDA support...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 324.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 247.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 240.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 293.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 277.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [7]:
from llama_cpp import Llama
import torch
print('⏳ Loading model from Drive into GPU VRAM...')
print('   (Reading 26.5 GB from Drive may take 2-4 min)\n')
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,   # All layers on GPU
    n_ctx=65536,        # Context window
    n_batch=512,
    verbose=False,
)
used_vram = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded!')
print(f'   VRAM used: {used_vram:.1f} GB / {total_vram:.1f} GB')
print(f'   VRAM free: {total_vram - used_vram:.1f} GB (available for context)')

⏳ Loading model from Drive into GPU VRAM...
   (Reading 26.5 GB from Drive may take 2-4 min)



llama_context: n_ctx_seq (65536) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


Model loaded!
   VRAM used: 0.0 GB / 42.4 GB
   VRAM free: 42.4 GB (available for context)


In [ ]:
def chat(    user_message,    system='You are a helpful assistant.',    max_tokens=1024,    temperature=0.7,    thinking=True,       # True = slower but deeper reasoning):    messages = [        {'role': 'system', 'content': system + (' /no_think' if not thinking else '')},        {'role': 'user',   'content': user_message},    ]    response = llm.create_chat_completion(        messages=messages,        max_tokens=max_tokens,        temperature=temperature,        stop=['<|im_end|>'],    )    return response['choices'][0]['message']['content']reply = chat('Explain greens theorem from scratch')print(f'\nAssistant: {reply}')


Assistant: Here's a thinking process:

1.  **Understand User Request:** The user wants an explanation of Green's Theorem "from scratch." This means I need to start from basic concepts, build up intuition, state the theorem clearly, explain its components, provide examples/interpretations, and maybe touch on its significance and limitations. No prior advanced knowledge should be assumed.

2.  **Identify Key Concepts to Cover:**
   - What is Green's Theorem? (High-level overview)
   - Prerequisites/Foundations: Line integrals, vector fields, double integrals, regions in the plane
   - Statement of the theorem (mathematical form)
   - Intuition/Geometric meaning (circulation vs. curl, flux vs. divergence)
   - Simple example to illustrate
   - Conditions/assumptions (smoothness, orientation, simply connected regions)
   - Why it's important/applications
   - Connection to other theorems (Stokes', Divergence Theorem)
   - Keep it accessible but mathematically precise

3.  **Structure the 

In [8]:
DATA_PATH = "/content/drive/MyDrive/neweraugmented_mimic_admission_only.csv"

df = pd.read_csv(DATA_PATH)

print(f"Data loaded: {df.shape[0]} rows")

Data loaded: 125 rows


In [9]:
df.head()

,HADM_ID,topic,ground_truth,SHORT_TITLE,LONG_TITLE,tokens,cleaned_text,augmentation_type,is_augmented,counterfactual_flag,admission_text
0,197345,neoplasm,1976,Sec mal neo peritoneum,Secondary malignant neoplasm of retroperitoneu...,1179,Sex: F Service: MEDICINE Allergies: Compazine ...,negated,True,NaN,Sex: F Service: MEDICINE Allergies: Compazine ...
1,197345,neoplasm,1976,Sec mal neo peritoneum,Secondary malignant neoplasm of retroperitoneu...,1179,Sex: [MASK] Service: MEDICINE Allergies: Compa...,random_masked,True,NaN,Sex: [MASK] Service: MEDICINE Allergies: Compa...
2,194302,neoplasm,1970,Secondary malig neo lung,Secondary malignant neoplasm of lung,614,Sex: M Service: BLUE SURGERY HISTORY OF THE PR...,negated,True,NaN,Sex: M Service: BLUE SURGERY HISTORY OF THE PR...
3,194302,neoplasm,1970,Secondary malig neo lung,Secondary malignant neoplasm of lung,614,Sex: M Service: [MASK] SURGERY [MASK] [MASK] T...,random_masked,True,NaN,Sex: M Service: [MASK] SURGERY [MASK] [MASK] T...
4,107691,neoplasm,22801,Hemangioma skin,Hemangioma of skin and subcutaneous tissue,1420,Sex: F Service: Neonatology IDENTIFICATION: is...,negated,True,NaN,Sex: F Service: Neonatology IDENTIFICATION: is...


In [ ]:
EXPERT_EXAMPLE = {    'text': "elderly male with multiple comorbidities including chf, diabetes, peripheral vascular disease, admitted with hypotension and decreased urine output, wound cultures positive for mrsa and pseudomonas, poor nutritional status, chronic non-healing ulcers",    'label': "Reasoning:\n- Chronic non-healing wounds indicate poor tissue repair — hallmark of protein-energy malnutrition\n- MRSA and Pseudomonas are opportunistic, thrive when immune system is compromised by malnutrition\n- Principal diagnosis is Malnutrition (protein-energy, unspecified)\n"}COUNTERFACTUAL_EXAMPLE = {    'text': EXPERT_EXAMPLE['text'],    'label': "Reasoning:\n- Patient presents with hypotension and decreased urine output suggesting volume depletion\n- Septic source from wound infections driving hemodynamic instability\n- Principal diagnosis is Acute Kidney Injury secondary to septic shock\n"}test_row   = df.iloc[0]hadm_id    = test_row['HADM_ID']negated_hx_text = df[        (df['HADM_ID'] == hadm_id) &        (df['augmentation_type'] == 'negated_hx')    ]['cleaned_text'].values[0]    negated_ruled_text = df[        (df['HADM_ID'] == hadm_id) &        (df['augmentation_type'] == 'negated_ruled_out')    ]['cleaned_text'].values[0]    negated_hedge_text = df[        (df['HADM_ID'] == hadm_id) &        (df['augmentation_type'] == 'negated_hedge')    ]['cleaned_text'].values[0]masked_text = df[    (df['HADM_ID'] == hadm_id) &    (df['augmentation_type'] == 'random_masked')]['cleaned_text'].values[0]conditions = [        ("0-shot",            test_row['cleaned_text'], None),        ("1-shot",            test_row['cleaned_text'], EXPERT_EXAMPLE),        ("counterfactual",    test_row['cleaned_text'], COUNTERFACTUAL_EXAMPLE),        ("negated_hx",        negated_hx_text,          None),        ("negated_ruled_out", negated_ruled_text,       None),        ("negated_hedge",     negated_hedge_text,       None),        ("random_masked",     masked_text,              None),    ]SYSTEM_PROMPT = """You are assisting a clinical NLP research study using de-identified patient dataRead the following clinical notes carefully and reason step by step:1. What are the key clinical findings?2. Are there any inconsistencies or conflicting information in the notes?3. Based on your reasoning, what is the most supported principal diagnosis?Think through each step before giving your Principal Diagnosis."""def get_prediction(messages, thinking=False, max_new_tokens=3072):    """    Accepts messages in your original format:    - No system role — system prompt is embedded in first user turn    - Content is list of dicts: [{"type": "text", "text": "..."}]    - Handles 0-shot and few-shot (user/assistant turns)    """    # Convert your content format [{"type": "text", "text": "..."}]    # to plain strings that llama-cpp expects    normalized = []    for msg in messages:        content = msg["content"]        if isinstance(content, list):            text = " ".join(block["text"] for block in content if block["type"] == "text")        else:            text = content  # already a string        # Inject /no_think into the FIRST user message if not thinking        if not thinking and msg["role"] == "user" and not normalized:            text = "/no_think\n\n" + text        normalized.append({"role": msg["role"], "content": text})    response = llm.create_chat_completion(        messages=normalized,        max_tokens=max_new_tokens,        temperature=0.0,       # greedy, same as do_sample=False        top_p=1.0,        stop=["<|im_end|>"],    )    output = response["choices"][0]["message"]["content"]    # Strip <think>...</think> if thinking=True but only want final answer    if thinking:        import re        output = re.sub(r"<think>.*?</think>", "", output, flags=re.DOTALL).strip()    return output# ── Run all conditions ────────────────────────────────────────────────────────for cond_name, input_text, example in conditions:    print(f"\n── condition: {cond_name} ──────────────────")    if example is not None:        messages = [            {"role": "user",      "content": [{"type": "text", "text": SYSTEM_PROMPT + f"\n\nClinical note:\n{example['text']}"}]},            {"role": "assistant", "content": [{"type": "text", "text": example['label']}]},            {"role": "user",      "content": [{"type": "text", "text": f"Clinical note:\n{input_text}"}]},        ]    else:        messages = [            {"role": "user", "content": [{"type": "text", "text": SYSTEM_PROMPT + f"\n\nClinical note:\n{input_text}"}]},        ]    start_time = time.time()    answer = get_prediction(messages, thinking=False)  # flip to True for deep reasoning    elapsed_time = time.time() - start_time    print(f"HADM_ID: {hadm_id} | topic: {test_row['topic']}")    print(f"Generation took {elapsed_time:.2f} seconds.")    print(f"Answer: {answer}")


── condition: 0-shot ──────────────────
HADM_ID: 197345 | topic: neoplasm
Generation took 35.74 seconds.
Answer: Here's a thinking process:

1.  **Analyze User Input:**
   - **Role:** Clinical NLP research assistant using de-identified patient data.
   - **Task:** Read clinical notes, reason step-by-step on:
     1. Key clinical findings
     2. Inconsistencies/conflicting information
     3. Most supported principal diagnosis
   - **Input Data:** A detailed clinical note for a 53yo F with metastatic breast cancer, presenting with abdominal pain, hypoxia, hypotension, tachycardia. Includes HPI, PMH, Social/Family Hx, Physical Exam, Pertinent Results (EKG, Echo, CT Torso), Hospital Course, Discharge info.
   - **Constraint:** `/no_think` - I must output the reasoning and final answer directly without internal monologue or hidden steps. I will structure the response clearly according to the prompt's 3 questions.

2.  **Deconstruct the Clinical Note (Mental Extraction):**
   - *Demograph

ให้เห็นว่า max_tokens= 3072 อาจจะไม่พอที่จะ generate respond ครบ เลยจะขอเลื่อนขึ้นไปเป็น 4096 ซึ่งลองเเล้วยังไม่พอ เลยเลื่อนเป็น 5012เเต่ที่สังเกตคือ time used per respond คือเฉลี่ยเเบบกะๆเอาได้ 35 sec ซึ่งเร็วกว่า dsk-distil-llama 8b หรือ meggemma 1.5 - 4b ที่ปกติใช้ 2 นาที

In [11]:
OUTPUT_PATH     = "/content/drive/MyDrive/alterqwen3_6_results.csv"
DATA_PATH       = "/content/drive/MyDrive/neweraugmented_mimic_admission_only.csv"
INJECTIONS_PATH = "/content/drive/MyDrive/patient_injection.json"

SYSTEM_PROMPT = """You are assisting a clinical NLP research study using de-identified patient data
Read the following clinical notes carefully and reason step by step:
1. What are the key clinical findings?
2. Are there any inconsistencies or conflicting information in the notes?
3. Based on your reasoning, what is the most supported principal diagnosis?
Think through each step before giving your Principal Diagnosis."""

EXPERT_EXAMPLE = {
    'text': "elderly male with multiple comorbidities including chf, diabetes, peripheral vascular disease, admitted with hypotension and decreased urine output, wound cultures positive for mrsa and pseudomonas, poor nutritional status, chronic non-healing ulcers",
    'label': "Reasoning:\n- Chronic non-healing wounds indicate poor tissue repair — hallmark of protein-energy malnutrition\n- MRSA and Pseudomonas are opportunistic, thrive when immune system is compromised by malnutrition\n- Principal diagnosis is Malnutrition (protein-energy, unspecified)\n",
}

INJECTION_TEMPLATE = (
    "[PRELIMINARY CHART REVIEW]\n"
    "Note: Based on initial triage, {injection}\n"
    "[END PRELIMINARY REVIEW]\n\n"
)

with open(INJECTIONS_PATH) as _f:
    patient_injections = json.load(_f)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} rows | augmentation types: {df['augmentation_type'].unique().tolist()}")


def get_prediction(messages, thinking=False, max_new_tokens=5012):
    normalized = []
    for msg in messages:
        content = msg["content"]
        if isinstance(content, list):
            text = " ".join(block["text"] for block in content if block["type"] == "text")
        else:
            text = content
        normalized.append({"role": msg["role"], "content": text})
    response = llm.create_chat_completion(
        messages=normalized,
        max_tokens=max_new_tokens,
        temperature=0.0,
        top_p=1.0,
        stop=["<|im_end|>"],
    )
    return response["choices"][0]["message"]["content"]


def _get_aug(hadm_id, aug_type):
    rows = df[(df['HADM_ID'] == hadm_id) & (df['augmentation_type'] == aug_type)]
    if len(rows) == 0:
        raise ValueError(f"No row for HADM {hadm_id} / augmentation_type='{aug_type}'")
    return rows['cleaned_text'].values[0]


# ── Incremental: skip already-done HADM_IDs ────────────────────────────────────
existing      = pd.read_csv(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else pd.DataFrame()
done_hadm_ids = set(existing['HADM_ID'].unique()) if len(existing) > 0 else set()

# Use 'negated' rows as the base (one per HADM_ID)
unique_per_topic = df[df['augmentation_type'] == 'negated'].drop_duplicates(subset=['topic', 'HADM_ID'])
sampled_ids = (
    unique_per_topic[~unique_per_topic['HADM_ID'].isin(done_hadm_ids)]
    .groupby('topic').head(1)['HADM_ID'].tolist()
)
df_todo = df[df['HADM_ID'].isin(sampled_ids) & (df['augmentation_type'] == 'negated')].copy()
print(f"Already done: {len(done_hadm_ids)} | To run: {len(df_todo)} HADM_IDs × 7 conditions = {len(df_todo) * 7} calls")

# ── Main loop ───────────────────────────────────────────────────────────────────
for i, (_, test_row) in enumerate(df_todo.iterrows(), start=1):
    hadm_id   = test_row['HADM_ID']
    base_text = test_row['cleaned_text']

    negated_hx_text    = _get_aug(hadm_id, 'negated_hx')
    negated_ruled_text = _get_aug(hadm_id, 'negated_ruled_out')
    negated_hedge_text = _get_aug(hadm_id, 'negated_hedge')
    masked_text        = _get_aug(hadm_id, 'random_masked')

    patient_injection = INJECTION_TEMPLATE.format(
    injection=patient_injections[str(int(hadm_id))])

    conditions = [
        ("0-shot",            base_text,             "0-shot"),
        ("1-shot",            base_text,             "1-shot"),
        ("counterfactual",    base_text,             "strong_injection"),
        ("negated_hx",        negated_hx_text,       "0-shot"),
        ("negated_ruled_out", negated_ruled_text,    "0-shot"),
        ("negated_hedge",     negated_hedge_text,    "0-shot"),
        ("random_masked",     masked_text,           "0-shot"),
    ]

    hadm_results = []
    for cond_name, input_text, cond_type in conditions:
        print(f"\n[{i}/{len(df_todo)}] HADM_ID: {hadm_id} | topic: {test_row['topic']} | condition: {cond_name}")

        if cond_type == "1-shot":
            messages = [
                {"role": "user",      "content": [{"type": "text", "text": SYSTEM_PROMPT + f"\n\nClinical note:\n{EXPERT_EXAMPLE['text']}"}]},
                {"role": "assistant", "content": [{"type": "text", "text": EXPERT_EXAMPLE['label']}]},
                {"role": "user",      "content": [{"type": "text", "text": f"Clinical note:\n{input_text}"}]},
            ]
        elif cond_type == "strong_injection":
            injected_text = patient_injection + f"Clinical note:\n{input_text}"
            messages = [
                {"role": "user", "content": [{"type": "text", "text": SYSTEM_PROMPT + f"\n\n{injected_text}"}]},
            ]
        else:
            messages = [
                {"role": "user", "content": [{"type": "text", "text": SYSTEM_PROMPT + f"\n\nClinical note:\n{input_text}"}]},
            ]

        start_time = time.time()
        answer     = get_prediction(messages, thinking=False)
        elapsed    = time.time() - start_time
        print(f"  Generation took {elapsed:.2f} seconds.")
        print(f"  Answer: {answer}")

        hadm_results.append({
            'HADM_ID'      : hadm_id,
            'topic'        : test_row['topic'],
            'ground_truth' : test_row['ground_truth'],
            'SHORT_TITLE'  : test_row['SHORT_TITLE'],
            'LONG_TITLE'   : test_row['LONG_TITLE'],
            'condition'    : cond_name,
            'final_answer' : answer,
        })
        time.sleep(0.5)

    existing = pd.concat([existing, pd.DataFrame(hadm_results)], ignore_index=True)
    existing.to_csv(OUTPUT_PATH, index=False)
    print(f"  >> Saved {len(existing)} rows total after HADM {hadm_id}")

print(f"\nAll done. {len(existing)} rows in {OUTPUT_PATH}")


Loaded 125 rows | augmentation types: ['negated', 'random_masked', 'negated_hx', 'negated_ruled_out', 'negated_hedge']
Already done: 0 | To run: 5 HADM_IDs × 7 conditions = 35 calls

[1/5] HADM_ID: 197345 | topic: neoplasm | condition: 0-shot
  Generation took 55.40 seconds.
  Answer: Here's a thinking process:

1.  **Analyze User Input:**
   - **Role:** Clinical NLP research assistant
   - **Task:** Read clinical notes, reason step-by-step on:
     1. Key clinical findings
     2. Inconsistencies/conflicting information
     3. Most supported principal diagnosis
   - **Input Data:** A detailed clinical note for a 53yo F with metastatic breast cancer, presenting with abdominal pain, hypoxia, hypotension, tachycardia. Includes HPI, PMH, Social/Family Hx, Physical Exam, Pertinent Results (EKG, Echo, CT Torso), Hospital Course, Discharge info.
   - **Output Requirement:** Step-by-step reasoning followed by the Principal Diagnosis.

2.  **Step 1: Identify Key Clinical Findings:**
   - *Dem

In [ ]:
!pip show torchprint("-" * 40)!nvcc --versionprint("-" * 40)try:    import torch    print("PyTorch Version:", torch.__version__)    print("CUDA Available:", torch.cuda.is_available())    if torch.cuda.is_available():        print("CUDA Version built with PyTorch:", torch.version.cuda)except ImportError as e:    print(f"Failed to import PyTorch. Error: {e}")

Name: torch
Version: 2.10.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision
----------------------------------------
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release